In [89]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

In [90]:
data = pd.read_excel("Data/dataset_comp_1222_without_ceg_ogn.xlsx")

## 1) Calculate historical returns in the period 2018-2020:

In [91]:
# # Define the tickers and date range
# tickers = list(data['SYMBOL'].values) # Add more tickers as needed
# start_date = "2015-11-01"
# end_date = "2021-01-01"

# # Download the data
# yahoo_data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bt = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bt[tickers[0]] = yahoo_data['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bt[ticker] = yahoo_data[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# print(adj_close_bt.head())

12 Failed downloads:
['SBNY', 'CEG', 'OGN']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1446350400, endDate = 1609477200")')
['MRO', 'CTLT', 'PXD', 'WRK', 'DISH', 'ATVI', 'SIVBQ']: YFTzMissingError('possibly delisted; no timezone found')
['DFS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01)')

In [92]:
# #print(data.loc[data['SYMBOL'] == 'BF.B'])
# # Define the tickers and date range
# tickers = ['BF-B'] # Add more tickers as needed
# start_date = "2015-11-01"
# end_date = "2021-01-01"

# # Download the data
# data_bfb = yf.download(tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=False, progress=False)

# # Extract adjusted close prices
# adj_close_bfb = pd.DataFrame()

# # If only one ticker, yfinance doesn't use a multi-indexed DataFrame, handle accordingly
# if len(tickers) == 1:
#     adj_close_bfb[tickers[0]] = data_bfb['BF-B']['Adj Close']
# else:
#     for ticker in tickers:
#         try:
#             adj_close_bfb[ticker] = data_bfb[ticker]['Adj Close']
#         except KeyError:
#             print(f"No data found for {ticker}.")

# # Display the result
# #print(adj_close_bfb.head())
# adj_close_bt['BF.B'] = adj_close_bfb.values

In [93]:
# adj_close_bt.to_excel("Data/adj_price_yahoo_comp_1222_historical.xlsx")

In [94]:
adj_close_bt = pd.read_excel("Data/adj_price_yahoo_comp_1222_historical.xlsx")
adj_close_bt.index = adj_close_bt.Date
adj_close_bt.drop(columns='Date', inplace=True)


'CEG', 'OGN'

['DFS']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-11-01 -> 2021-01-01) (Yahoo error = "No data found, symbol may be delisted")')

     

In [95]:
price_historical = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='CLOSE PRICE', header = 4)

price_historical.columns = price_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()

price_historical = price_historical.iloc[217:1566]

div_rate = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='DIV RATE', header = 4)
div_rate.columns = div_rate.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_rate = div_rate.iloc[217:1566]

div_date_historical = pd.read_excel('Data/price_div_comp_1222_historical.xlsm', sheet_name='DIV DATE', header = 4)
div_date_historical.columns = div_date_historical.columns.str.replace(r'\(.*?\)', '', regex=True).str.strip()
div_date_historical = div_date_historical.iloc[217:1566]

div_date_historical.columns = price_historical.columns
div_rate.columns = price_historical.columns

price_historical.index = price_historical.Code
div_rate.index = div_rate.Code
div_date_historical.index = div_date_historical.Code

price_historical = price_historical.iloc[:, 1:]
div_rate = div_rate.iloc[:, 1:]
div_date_historical = div_date_historical.iloc[:, 1:]

# --- Step 1: Calculate adjustment factors
adj_factors = pd.DataFrame(1.0, index=price_historical.index, columns=price_historical.columns)

for company in price_historical.columns:
    for i in range(1, len(price_historical)):
        date = price_historical.index[i]
        prev_date = price_historical.index[i - 1]

        # If ex-dividend happens on this day
        if pd.notna(div_date_historical.at[date, company]):
            div = div_rate.at[date, company]
            price_prev = price_historical.at[prev_date, company]
            if price_prev and price_prev != 0:
                factor = (price_prev - div) / price_prev
                adj_factors.at[date, company] = factor

# --- Step 2: Calculate cumulative adjustment factors in reverse (like Yahoo)
cum_factors = adj_factors.iloc[::-1].cumprod().iloc[::-1]

# --- Step 3: Build adjusted prices
adjusted_prices_calculated = price_historical * cum_factors

#print(data.loc[data['SYMBOL'] == 'WRK', 'NAME'])
#print(div_rate['96699P'].unique())
#print(price_historical['96699P'])
#print(adjusted_prices_calculated['96699P'])
adj_close_bt['WRK'] = adjusted_prices_calculated.loc[adj_close_bt.index, '96699P'].values

#print(data.loc[data['SYMBOL'] == 'CTLT', 'NAME'])
#print(div_rate['8866F3'].unique())
#print(price_historical['8866F3'])
#print(adjusted_prices_calculated['8866F3'])
adj_close_bt['CTLT'] = adjusted_prices_calculated.loc[adj_close_bt.index, ['8866F3']].values

#print(data.loc[data['SYMBOL'] == 'MRO', 'NAME'])
#print(div_rate['544682'].unique())
#print(price_historical['544682'])
#print(adjusted_prices_calculated['544682'])
adj_close_bt['MRO'] = adjusted_prices_calculated.loc[adj_close_bt.index, '544682'].values

#print(data.loc[data['SYMBOL'] == 'PXD', 'NAME'])
#print(div_rate['895705'].unique())
#print(price_historical['895705'])
adj_close_bt['PXD'] = adjusted_prices_calculated.loc[adj_close_bt.index, '895705'].values

#print(data.loc[data['SYMBOL'] == 'SBNY', ['NAME','TYPE']])
#print(div_rate['28709C'].unique())
#print(price_historical['28709C'])
adj_close_bt['SBNY'] = adjusted_prices_calculated.loc[adj_close_bt.index, '28709C'].values

#print(data.loc[data['SYMBOL'] == 'ATVI', ['NAME','TYPE']])
#print(div_rate['312367'].unique())
#print(price_historical['312367'])
adj_close_bt['ATVI'] = adjusted_prices_calculated.loc[adj_close_bt.index, '312367'].values

#print(data.loc[data['SYMBOL'] == 'DISH', ['NAME','TYPE']])
#print(div_rate['135448'].unique())
#print(price_historical['135448'])
adj_close_bt['DISH'] = adjusted_prices_calculated.loc[adj_close_bt.index, '135448'].values

#print(data.loc[data['SYMBOL'] == 'SIVBQ', ['NAME','TYPE']])
#print(div_rate['518628'].unique())
#print(price_historical['518628'])
adj_close_bt['SIVBQ'] = adjusted_prices_calculated.loc[adj_close_bt.index, '518628'].values

print(div_rate['50642F'].unique())
print(price_historical['50642F'])
adj_close_bt['DFS'] = adjusted_prices_calculated.loc[adj_close_bt.index, '50642F'].values

[ nan 0.28 0.3  0.35 0.4  0.44]
Code
2015-11-02    56.28
2015-11-03    56.35
2015-11-04    56.37
2015-11-05    56.91
2015-11-06    57.61
              ...  
2020-12-25    88.29
2020-12-28    88.29
2020-12-29    88.02
2020-12-30    89.30
2020-12-31    90.53
Name: 50642F, Length: 1349, dtype: float64


In [96]:
# ceg and ogn already excluded

In [97]:
#right now my log returns go from 15 dec 2020 to 15 dec 2021 (so the data goes back from 15 nov 2020)
# I could back test three years before so :

# 15 nov 2017 to 15 nov 2018
# 15 nov 2018 to 15 nov 2019
# 15 nov 2019 to 15 nov 2020

In [98]:
from datetime import datetime
adj_close_bt_above_2018 = adj_close_bt.loc[adj_close_bt.index > datetime(2017, 11, 15)]
cols_with_nans = adj_close_bt_above_2018.columns[adj_close_bt_above_2018.isna().any()]
print("Columns with NaNs:", cols_with_nans.tolist())

nan_columns = adj_close_bt_above_2018.loc[:, adj_close_bt_above_2018.isna().any(axis=0)]

for col in nan_columns.columns:
    first_valid = nan_columns[col].first_valid_index()
    print(f"First non-NaN in column '{col}': {first_valid}") # these stocks had later IPO

Columns with NaNs: ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA', 'OGN', 'CEG', 'VICI']
First non-NaN in column 'DAY': 2018-04-26 00:00:00
First non-NaN in column 'MRNA': 2018-12-07 00:00:00
First non-NaN in column 'DOW': 2019-03-20 00:00:00
First non-NaN in column 'FOXA': 2019-03-12 00:00:00
First non-NaN in column 'CARR': 2020-03-19 00:00:00
First non-NaN in column 'OTIS': 2020-03-19 00:00:00
First non-NaN in column 'CTVA': 2019-05-24 00:00:00
First non-NaN in column 'OGN': None
First non-NaN in column 'CEG': None
First non-NaN in column 'VICI': 2018-01-02 00:00:00


VICI is okay, OGN and CEG I have to remove them in any case. For the rest I can take the optimized portfolio, and exclude these 7 stocks and then I can see how does it change in practice the optimised portfolio. After doing this I can evaluate the portfolio back in time. 

In [99]:

nan_backtest = ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA', 'VICI']
for stock in nan_backtest:
    stock_type_miss = data.loc[data['SYMBOL'] == stock, 'TYPE'].values
    if price_historical[stock_type_miss].first_valid_index() < nan_columns[stock].first_valid_index():
        print(stock, price_historical[stock_type_miss].first_valid_index())

VICI 2017-10-17 00:00:00


In [100]:
vici = price_historical['9174A0']
vici = vici[vici.index >= datetime(2017, 12, 1)]

In [101]:
adj_close_bt = adj_close_bt[(adj_close_bt.index >= datetime(2017, 12, 1))]

In [102]:
adj_close_bt['VICI'] = vici

In [103]:
adj_close_bt = adj_close_bt.drop(columns=['OGN', 'CEG', 'DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA'])

In [104]:
def calculate_monthly_returns(df):
    # Clean data: replace zeros and drop fully NaN rows
    df = df.replace(0.0, np.nan).dropna(how="all")
    
    # Resample to monthly frequency using last available price in each month
    df_monthly = df.resample("M").last()
    
    # Calculate monthly simple returns (percentage change)
    df_returns = df_monthly.pct_change().dropna()
    
    # Optional: set index to mid-month for visualization consistency
    df_returns.index = df_returns.index.map(lambda x: x.replace(day=15))
    
    return df_returns

In [105]:
historical_returns_2018_2020 = calculate_monthly_returns(adj_close_bt)

/var/folders/n9/cy4s67js3v58hhczg4lxj5jr0000gq/T/ipykernel_91944/476870519.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly = df.resample("M").last()


In [124]:
historical_returns_2018_2020

,AMZN,ABT,AES,IBM,AMD,ADBE,ARE,APD,ALK,BXP,...,HPE,PYPL,VICI,HPQ,DXC,WEC,MNST,LIN,SBAC,CHTR
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-15,0.240639,0.094397,0.079492,0.067006,0.336576,0.139922,-0.006815,0.026145,-0.105836,-0.048604,...,0.142061,0.158924,0.073171,0.109948,0.048999,-0.032064,0.078053,0.044027,0.068193,0.122902
2018-02-15,0.042429,-0.029440,-0.059689,-0.038703,-0.118632,0.046906,-0.064688,-0.045020,-0.013944,-0.039124,...,0.133537,-0.069269,-0.111364,0.003002,0.030035,-0.059632,-0.071230,-0.072698,-0.098739,-0.093625
2018-03-15,-0.043049,-0.006796,0.045998,-0.015401,-0.170107,0.033233,0.037164,-0.004020,-0.039380,0.043501,...,-0.052757,-0.044579,-0.062916,-0.057349,-0.017901,0.046395,-0.097207,-0.031161,0.086794,-0.089814
2018-04-15,0.082075,-0.025203,0.088007,-0.055205,0.082587,0.025546,-0.002562,0.020499,0.047934,-0.014689,...,-0.027936,-0.016607,-0.007642,-0.019617,0.025167,0.025199,-0.038630,0.056965,-0.062544,-0.128301
2018-05-15,0.040539,0.058489,0.041666,-0.014358,0.261948,0.124910,0.002810,-0.005423,-0.058276,0.002965,...,-0.106158,0.099987,0.065457,0.025128,-0.106249,-0.008765,-0.069818,0.024521,-0.013481,-0.037782
2018-06-15,0.043065,-0.008776,0.051765,-0.011393,0.091770,-0.021943,0.017588,-0.028340,-0.006907,0.036650,...,-0.034528,0.014622,0.065565,0.035956,0.013958,0.023753,0.120016,0.017357,0.044600,0.123238
2018-07-15,0.045676,0.079433,-0.003729,0.037437,0.222815,0.003568,0.010066,0.054196,0.040404,0.000877,...,0.056810,-0.013570,-0.014050,0.017189,0.051234,0.026605,0.047469,0.059121,-0.041606,0.038778
2018-08-15,0.132364,0.019835,0.017303,0.021614,0.373159,0.076958,0.007141,0.012913,0.079483,0.039194,...,0.070596,0.124057,0.027518,0.068024,0.074935,0.026686,0.014495,-0.050805,-0.019084,0.019108
2018-09-15,-0.004824,0.097547,0.040119,0.032291,0.227255,0.024439,-0.012444,0.011171,0.020299,-0.048977,...,-0.006468,-0.048630,0.033955,0.051364,0.028823,-0.012134,-0.042864,0.016056,0.034787,0.049871


# 2) I take the benchmark portfolio weights of dec 22 and let them drift naturally (buy-and-hold portfolio)

In [120]:
import pickle
with open('Data/optimal_portfolios_1222_without_ceg_ogn.pkl', 'rb') as f:
    optimal_portfolios_shrink = pickle.load(f)


In [122]:
exclude_list = ['DAY', 'MRNA', 'DOW', 'FOXA', 'CARR', 'OTIS', 'CTVA']

# Copy to avoid modifying in place
adjusted_portfolios = {}

for sector, data in optimal_portfolios_shrink.items():
    stock_labels = data['stock_labels']
    w_b = data['w_b_vec']
    w_opt = data['w_opt']
    
    # Create a mask for stocks not in the exclude list
    keep_mask = ~stock_labels.isin(exclude_list)
    
    # Apply mask to weights and stock_labels
    w_b_new = w_b[keep_mask]
    w_opt_new = w_opt[keep_mask]
    labels_new = stock_labels[keep_mask]
    
    # Re-normalize weights to sum to 1
    w_b_new /= w_b_new.sum()
    w_opt_new /= w_opt_new.sum()
    
    # Store in new dictionary
    adjusted_portfolios[sector] = {
        'w_b_vec': w_b_new,
        'w_opt': w_opt_new,
        'stock_labels': labels_new
    }


In [123]:
for sector, data in adjusted_portfolios.items():
    print(f"{sector} — benchmark sum: {data['w_b_vec'].sum():.4f}, opt sum: {data['w_opt'].sum():.4f}")


Consumer Discretionary — benchmark sum: 1.0000, opt sum: 1.0000
Health Care — benchmark sum: 1.0000, opt sum: 1.0000
Utilities — benchmark sum: 1.0000, opt sum: 1.0000
Information Technology — benchmark sum: 1.0000, opt sum: 1.0000
Real Estate — benchmark sum: 1.0000, opt sum: 1.0000
Materials — benchmark sum: 1.0000, opt sum: 1.0000
Industrials — benchmark sum: 1.0000, opt sum: 1.0000
Financials — benchmark sum: 1.0000, opt sum: 1.0000
Energy — benchmark sum: 1.0000, opt sum: 1.0000
Communication Services — benchmark sum: 1.0000, opt sum: 1.0000
Consumer Staples — benchmark sum: 1.0000, opt sum: 1.0000


In [125]:
import pandas as pd
import numpy as np

def compute_buy_and_hold_returns(returns_df, weights, stock_labels):
    """
    Compute buy-and-hold (non-rebalanced) portfolio returns.
    
    Args:
        returns_df: DataFrame with stock returns (monthly), index = date, columns = tickers
        weights: np.array of initial weights (must match stock_labels)
        stock_labels: list or Index of tickers in this sector

    Returns:
        Series of portfolio returns over time (monthly)
    """
    # Subset return data for the sector
    sector_returns = returns_df[stock_labels]
    
    # Compute cumulative returns per stock
    cumulative_returns = (1 + sector_returns).cumprod()
    
    # Multiply by initial weights
    weighted_growth = cumulative_returns.multiply(weights, axis=1)
    
    # Total portfolio value each month
    portfolio_value = weighted_growth.sum(axis=1)
    
    # Convert to monthly portfolio returns
    portfolio_returns = portfolio_value.pct_change().dropna()

    return portfolio_returns


In [126]:
buy_and_hold_sector_returns = {}

for sector, data in adjusted_portfolios.items():
    w_b = data['w_b_vec']                 # initial benchmark weights
    tickers = data['stock_labels']       # stock labels for the sector
    
    # Compute sector-level buy-and-hold benchmark returns
    sector_returns = compute_buy_and_hold_returns(
        historical_returns_2018_2020,
        weights=w_b,
        stock_labels=tickers
    )
    
    buy_and_hold_sector_returns[sector] = sector_returns


In [127]:
buy_and_hold_df = pd.DataFrame(buy_and_hold_sector_returns)


In [129]:
def compute_fixed_weight_returns(returns_df, weights, stock_labels):
    """
    Compute fixed-weight portfolio returns (decarbonised portfolio).
    
    Args:
        returns_df: DataFrame of monthly returns
        weights: np.array of optimised weights
        stock_labels: list or Index of stock tickers

    Returns:
        Series of portfolio returns over time
    """
    sector_returns = returns_df[stock_labels]
    portfolio_returns = sector_returns.dot(weights)
    return portfolio_returns


In [130]:
decarb_sector_returns = {}

for sector, data in adjusted_portfolios.items():
    w_opt = data['w_opt']
    tickers = data['stock_labels']
    
    sector_returns = compute_fixed_weight_returns(
        historical_returns_2018_2020,
        weights=w_opt,
        stock_labels=tickers
    )
    
    decarb_sector_returns[sector] = sector_returns


In [131]:
import numpy as np

sector_tracking_errors = {}

for sector in buy_and_hold_sector_returns:
    # Get returns
    r_b = buy_and_hold_sector_returns[sector]
    r_d = decarb_sector_returns[sector]
    
    # Align indices (just in case)
    common_index = r_b.index.intersection(r_d.index)
    active_returns = r_d.loc[common_index] - r_b.loc[common_index]
    
    # Monthly tracking error
    te_monthly = active_returns.std()
    
    # Annualised tracking error
    te_annualised = te_monthly * np.sqrt(12)
    
    # Store
    sector_tracking_errors[sector] = te_annualised


In [132]:
import pandas as pd

te_df = pd.DataFrame.from_dict(sector_tracking_errors, orient='index', columns=['Annualised Tracking Error'])
te_df = te_df.sort_values(by='Annualised Tracking Error', ascending=False)


In [133]:
te_df

,Annualised Tracking Error
Consumer Discretionary,0.080601
Real Estate,0.042161
Financials,0.041663
Materials,0.039576
Health Care,0.039276
Industrials,0.037728
Energy,0.034336
Utilities,0.030864
Information Technology,0.028965
Consumer Staples,0.026495


Next steps: 
- Do it for multiple TEs to see if this rank order is preserved across different TE optimisation constraints
- Do it for the period 2023-2025 (half a year only)